In [ ]:
import pandas as pd
import re
from pathlib import Path

# Diretório base e utilitário para dados
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'Banco_de_Dados_PII3_AWS').exists() and (BASE_DIR.parent / 'Banco_de_Dados_PII3_AWS').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'Banco_de_Dados_PII3_AWS'
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretório de dados não encontrado: {DATA_DIR}")

def get_data_path(nome_arquivo):
    return DATA_DIR / nome_arquivo

In [2]:
def limpar_avaliacoes(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip()
    texto = re.sub(r'^[\W_]+', '', texto, flags=re.UNICODE)
    texto = re.sub(r'\s{2,}', ' ', texto)

    # Remove espacos antes de pontuacao e reduz repeticoes
    texto = re.sub(r'\s+([.,;:!?])', r'\1', texto)
    texto = re.sub(r'([.,;:!?])\1+', r'\1', texto)
    texto = re.sub(r'([.,;:!?])(?=[^\s])', r'\1 ', texto)

    # Padroniza abreviacoes comuns
    mapeamento = {
        r'\b(n[ãa]o|nã|nao)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ''
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break

        if original.isupper():
            return alvo.upper()
        if original and original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    texto = re.sub(r'(\.\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)
    texto = re.sub(r'[\s]+$', '', texto)
    return texto



In [3]:
def juntar_titulo_mensagem(df, coluna_titulo='review_comment_title', coluna_mensagem='review_comment_message'):
    if coluna_titulo not in df.columns or coluna_mensagem not in df.columns:
        print(f'Erro: colunas {coluna_titulo} ou {coluna_mensagem} nao encontradas.')
        return df

    df[coluna_titulo] = df[coluna_titulo].astype('string')
    df[coluna_mensagem] = df[coluna_mensagem].astype('string')

    def combinar(titulo, mensagem):
        titulo = str(titulo).strip() if pd.notna(titulo) else ''
        mensagem = str(mensagem).strip() if pd.notna(mensagem) else ''

        if titulo and mensagem:
            return f'{titulo} - {mensagem}'
        return titulo or mensagem

    df[coluna_mensagem] = df.apply(lambda row: combinar(row[coluna_titulo], row[coluna_mensagem]), axis=1)
    return df

In [4]:
def processar_csv(caminho_arquivo, nome_coluna):
    # aceita string ou Path
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo nao encontrado em {caminho}')
        return

    df = pd.read_csv(caminho)

    if nome_coluna in df.columns:
        print(f'Limpando espacos e padronizando a coluna {nome_coluna}...')
        df[nome_coluna] = df[nome_coluna].apply(limpar_avaliacoes)
        df.to_csv(caminho, index=False)
        print('Sucesso! Espacos duplos removidos e arquivo atualizado.')
    else:
        print(f'Erro: A coluna {nome_coluna} nao existe no arquivo CSV.')

In [ ]:
def remover_linhas_sem_review(caminho_arquivo, coluna='review_comment_message'):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo nao encontrado em {caminho}')
        return

    df = pd.read_csv(caminho)
    if coluna not in df.columns:
        print(f'Erro: A coluna {coluna} nao existe no arquivo CSV.')
        return

    antes = len(df)
    df[coluna] = df[coluna].astype('string')
    df = df[df[coluna].str.strip() != '']
    depois = len(df)
    removidas = antes - depois

    df.to_csv(caminho, index=False)
    print(f'Removidas {removidas} linhas sem valor em {coluna}.')
    return df

In [11]:
def apagar_coluna(caminho_arquivo, nome_coluna):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo nao encontrado em {caminho}')
        return

    df = pd.read_csv(caminho)
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} nao existe no arquivo CSV.')
        return

    df = df.drop(columns=[nome_coluna])
    df.to_csv(caminho, index=False)
    print(f'Coluna {nome_coluna} apagada com sucesso.')
    return df

In [8]:
# Exemplo de uso:
df = pd.read_csv(get_data_path('avaliacoes.csv'))
df['review_comment_title'] = df['review_comment_title'].apply(limpar_avaliacoes)
df['review_comment_message'] = df['review_comment_message'].apply(limpar_avaliacoes)
df = juntar_titulo_mensagem(df)
df.to_csv(get_data_path('avaliacoes.csv'), index=False)

In [ ]:
def converter_para_string(df, nome_coluna):
    """
    Converte uma coluna do tipo object para string.
    """
    if nome_coluna in df.columns:
        df[nome_coluna] = df[nome_coluna].astype("string")
        print(f"Coluna '{nome_coluna}' convertida para string.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

df = pd.read_csv(get_data_path('avaliacoes.csv'))

converter_para_string(df, 'order_id')

In [ ]:
apagar_coluna(get_data_path('avaliacoes.csv'), 'review_comment_title')

In [ ]:
def converter_para_datetime(df, nome_coluna):
    """
    Converte uma coluna do tipo object para datetime.
    """
    if nome_coluna in df.columns:
        # errors='coerce' transforma valores inválidos em nulos (NaT)
        df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
        print(f"Coluna '{nome_coluna}' convertida para datetime.")
    else:
        print(f"Erro: Coluna '{nome_coluna}' não encontrada.")
    return df

converter_para_datetime(df, 'review_answer_timestamp')